# Split feasibility

Compare provisional common subject folds across tasks. Set `COHORTS` to select years. Edit `DATA_DIR` below before running.

Cohorts use separate assignments shared across tasks. All subjects receive a fold; only labels `{0, 1, 2}` contribute class indicators. `-1` remains unresolved.

Requires Python 3.10+, NumPy, pandas, IPython.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable, List
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Data directory containing the 1year, 2year, and 3year folders.
DATA_DIR = Path(r"C:\Users\emanu\Documents\Progetti_python\SLD-slimGSGP\data")
COHORTS = None  # All years; use ["1year"] to select one cohort.

EDA_VALID_LABELS = (0, 1, 2)
DEFAULT_SEED = 42
EDA_CANDIDATE_SEEDS = (DEFAULT_SEED, DEFAULT_SEED + 1)
EDA_FOLD_COUNTS = (2, 3, 4, 5)
EDA_MIN_FOLD_SUPPORT = 5
EDA_EXPECTED_TASKS = {"1year": 21, "2year": 21, "3year": 18}

In [2]:
@dataclass
class TaskTable:
    """One source file; source is relative to its cohort directory."""

    cohort: str
    source: str
    frame: pd.DataFrame

    @property
    def task(self) -> str:
        return str(Path(self.source).with_suffix(""))


def natural_key(value: str) -> tuple:
    """Sort numeric filename components numerically, with deterministic ties."""
    return tuple(
        (1, int(part)) if part.isdigit() else (0, part.casefold())
        for part in re.split(r"(\d+)", value)
    ) + ((0, value),)


def load_tasks(data_dir: Path, cohorts: Iterable[str] | None = None) -> list[TaskTable]:
    """Discover CSVs in natural order; preserve missing IDs and unknown labels."""
    root = Path(data_dir).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(
            f"Set DATA_DIR to the data directory containing cohort folders: {root}"
        )
    selected = (
        list(cohorts)
        if cohorts is not None
        else [
            p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")
        ]
    )
    if len(selected) != len(set(selected)):
        raise ValueError("Select each cohort only once.")
    tables = []
    for cohort in sorted(selected, key=natural_key):
        directory = root / cohort
        if not directory.is_dir():
            raise FileNotFoundError(f"Cohort directory does not exist: {cohort}")
        files = sorted(
            (
                p
                for p in directory.rglob("*")
                if p.is_file() and p.suffix.lower() == ".csv"
            ),
            key=lambda p: natural_key(p.relative_to(directory).as_posix()),
        )
        if not files:
            raise FileNotFoundError(f"No task CSVs found in cohort: {cohort}")
        for path in files:
            frame = pd.read_csv(path, dtype="string")
            tables.append(
                TaskTable(cohort, path.relative_to(directory).as_posix(), frame)
            )
    if not tables:
        raise ValueError("No cohorts selected.")
    return tables


def cohort_tables(tables: list[TaskTable]) -> dict[str, list[TaskTable]]:
    result: dict[str, list[TaskTable]] = {}
    for table in tables:
        result.setdefault(table.cohort, []).append(table)
    return result


tables = load_tasks(DATA_DIR, cohorts=COHORTS)
pd.set_option("display.max_rows", 24)
pd.set_option("display.max_columns", 12)
print(
    "Loaded task files by cohort:",
    {c: len(g) for c, g in cohort_tables(tables).items()},
)

Loaded task files by cohort: {'1year': 21, '2year': 21, '3year': 18}


In [3]:
DEMOGRAPHIC_COLUMNS: List[str] = [
    "Gender",
    "Age",
    "Dominant_Hand",
]


def cohort_summary(tables: list[TaskTable]) -> pd.DataFrame:
    rows = []
    for cohort, group in cohort_tables(tables).items():
        subject_sets = []
        for table in group:
            ids = table.frame.get("Id", pd.Series(dtype="string")).dropna()
            subject_sets.append(set(ids[ids.astype("string").str.strip().ne("")]))
        combined = pd.concat([t.frame for t in group], ignore_index=True)
        conflicts = {}
        for column in DEMOGRAPHIC_COLUMNS:
            count = 0
            if {"Id", column}.issubset(combined):
                count = int(
                    combined.groupby("Id")[column].nunique(dropna=False).gt(1).sum()
                )
            conflicts[f"{column}_conflicting_subjects"] = count
        rows.append(
            dict(
                cohort=cohort,
                task_files=len(group),
                expected_tasks=EDA_EXPECTED_TASKS.get(cohort),
                rows=sum(len(t.frame) for t in group),
                subjects=len(set().union(*subject_sets)),
                subjects_in_every_task=len(set.intersection(*subject_sets)),
                identical_subject_sets=all(s == subject_sets[0] for s in subject_sets),
                **conflicts,
            )
        )
    return pd.DataFrame(rows)


def subject_label_matrix(tables: list[TaskTable]) -> pd.DataFrame:
    """Return one cohort's outer join; refuse ambiguous or missing identifiers.

    Matching IDs and demographics only support identity linkage. Researcher
    confirmation of acquisition identifiers is still needed before final folds.
    """
    if not tables or len({t.cohort for t in tables}) != 1:
        raise ValueError("A subject label matrix requires exactly one cohort.")
    columns = {}
    for table in tables:
        ids = table.frame["Id"].astype("string")
        if ids.isna().any() or ids.str.strip().eq("").any() or ids.duplicated().any():
            raise ValueError(
                f"Missing or duplicate IDs prevent a safe join in {table.task}."
            )
        if table.task in columns:
            raise ValueError("Duplicate source task keys prevent a safe join.")
        columns[table.task] = pd.Series(
            table.frame.Label.to_numpy(), index=ids, name=table.task
        )
    matrix = pd.concat(columns, axis=1)
    matrix.index.name = "Id"
    return matrix.sort_index()


def valid_labels(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where(numeric.isin(EDA_VALID_LABELS))


def theoretical_feasibility(
    tables: list[TaskTable],
    fold_counts: tuple[int, ...] = EDA_FOLD_COUNTS,
    min_support: int = EDA_MIN_FOLD_SUPPORT,
) -> pd.DataFrame:
    """Necessary support conditions use distinct subjects, never duplicate rows."""
    rows = []
    for cohort, group in cohort_tables(tables).items():
        matrix = subject_label_matrix(group).apply(valid_labels)
        for task in matrix:
            for k in fold_counts:
                if k < 2:
                    raise ValueError("Fold counts must be at least two.")
                for label in EDA_VALID_LABELS:
                    count = int(matrix[task].eq(label).sum())
                    rows.append(
                        dict(
                            cohort=cohort,
                            task=task,
                            folds=k,
                            label=label,
                            subject_count=count,
                            class_absent=count == 0,
                            every_test_fold_coverage_possible=count >= k,
                            unavoidable_missing_test_folds=max(0, k - count),
                            balanced_test_support_floor=count // k,
                            balanced_training_support_floor=count
                            - int(np.ceil(count / k)),
                            tiny_test_support=count // k < min_support,
                            tiny_training_support=count - int(np.ceil(count / k))
                            < min_support,
                            support_warning_threshold=min_support,
                        )
                    )
    return pd.DataFrame(rows)


cohort_overview = cohort_summary(tables)
print("Candidate fold counts:", EDA_FOLD_COUNTS, "Fixed seeds:", EDA_CANDIDATE_SEEDS)
print(
    "Descriptive minimum-support warning:",
    EDA_MIN_FOLD_SUPPORT,
    "subjects per class per partition",
)
display(cohort_overview)
feasibility_parts, feasibility_warnings = [], []
for cohort, group in cohort_tables(tables).items():
    try:
        feasibility_parts.append(theoretical_feasibility(group))
    except (ValueError, KeyError) as error:
        feasibility_warnings.append({"cohort": cohort, "reason": str(error)})
feasibility = (
    pd.concat(feasibility_parts, ignore_index=True)
    if feasibility_parts
    else pd.DataFrame()
)
if feasibility_warnings:
    display(pd.DataFrame(feasibility_warnings))

Candidate fold counts: (2, 3, 4, 5) Fixed seeds: (42, 43)
Descriptive minimum-support warning: 5 subjects per class per partition


,cohort,task_files,expected_tasks,rows,subjects,subjects_in_every_task,identical_subject_sets,Gender_conflicting_subjects,Age_conflicting_subjects,Dominant_Hand_conflicting_subjects
0,1year,21,21,2935,140,136,False,0,0,0
1,2year,21,21,3045,145,145,True,0,0,0
2,3year,18,18,2610,145,145,True,0,0,0


In [4]:
if not feasibility.empty:
    task_support = (
        feasibility.groupby(["cohort", "folds", "task"], sort=False)
        .agg(
            class_coverage_necessary_condition=(
                "every_test_fold_coverage_possible",
                "all",
            ),
            any_class_absent=("class_absent", "any"),
            smallest_class=("subject_count", "min"),
            tiny_test_support=("tiny_test_support", "any"),
            tiny_training_support=("tiny_training_support", "any"),
        )
        .reset_index()
    )
    display(
        task_support.groupby(["cohort", "folds"], sort=False).agg(
            tasks=("task", "size"),
            tasks_passing_necessary_condition=(
                "class_coverage_necessary_condition",
                "sum",
            ),
            tasks_with_absent_class=("any_class_absent", "sum"),
            worst_class_support=("smallest_class", "min"),
            tasks_with_tiny_test_support=("tiny_test_support", "sum"),
            tasks_with_tiny_training_support=("tiny_training_support", "sum"),
        )
    )
    display(
        feasibility.loc[
            feasibility.folds.eq(3) & ~feasibility.every_test_fold_coverage_possible,
            [
                "cohort",
                "task",
                "label",
                "subject_count",
                "unavoidable_missing_test_folds",
            ],
        ]
    )

tasks  tasks_passing_necessary_condition  \
cohort folds                                             
1year  2         21                                 19   
       3         21                                 19   
       4         21                                 18   
       5         21                                 17   
2year  2         21                                 15   
       3         21                                 14   
       4         21                                 13   
       5         21                                 13   
3year  2         18                                  9   
       3         18                                  7   
       4         18                                  5   
       5         18                                  5   

              tasks_with_absent_class  worst_class_support  \
cohort folds                                                 
1year  2                            1                    0   
       3                            1                    0   
       4                            1                    0   
       5                            1                    0   
2year  2                            4                    0   
       3                            4                    0   
       4                            4                    0   
       5                            4                    0   
3year  2                            9                    0   
       3                            9                    0   
       4                            9                    0   
       5                            9                    0   

              tasks_with_tiny_test_support  tasks_with_tiny_training_support  
cohort folds                                                                  
1year  2                                 7                                 7  
       3                                 9                                 4  
       4                                10                                 4  
       5                                13                                 4  
2year  2                                10                                10  
       3                                12                                 9  
       4                                15                                 9  
       5                                17                                 9  
3year  2                                15                                15  
       3                                16                                14  
       4                                18                                14  
       5                                18                                14

,cohort,task,label,subject_count,unavoidable_missing_test_folds
3,1year,Task_1,0,0,3
4,1year,Task_1,1,0,3
5,1year,Task_1,2,0,3
39,1year,Task_4,0,1,2
255,2year,Task_1,0,0,3
...,...,...,...,...,...
615,3year,Task_10,0,2,1
627,3year,Task_11,0,2,1
699,3year,Task_17,0,0,3
700,3year,Task_17,1,0,3


Each class needs at least K subjects for coverage in every test fold; joint coverage can still fail. The seeded greedy allocator balances task-class indicators and keeps fold sizes within one subject. Coverage alone does not establish reliable estimates.

In [5]:
def candidate_assignment(
    matrix: pd.DataFrame, folds: int, seed: int = DEFAULT_SEED
) -> pd.Series:
    """Greedy iterative multilabel allocation with hard balanced size capacities.

    Subjects carry binary task-class indicators. Allocate a remaining rare
    indicator first, then prefer the fold with greatest unmet indicator demand,
    breaking ties by aggregate demand, capacity, and seeded random choice.
    This heuristic does not certify joint coverage or statistical reliability.
    """
    if matrix.index.has_duplicates or matrix.index.isna().any():
        raise ValueError("Assignments require unique, nonmissing subject IDs.")
    if not 2 <= folds <= len(matrix):
        raise ValueError("Require 2 <= folds <= number of subjects.")
    valid = matrix.apply(valid_labels)
    indicators = np.column_stack(
        [
            valid[task].eq(label).fillna(False).to_numpy(dtype=bool)
            for task in valid
            for label in EDA_VALID_LABELS
        ]
    )
    rng = np.random.default_rng(seed)
    capacities = np.full(folds, len(matrix) // folds, dtype=int)
    capacities[rng.permutation(folds)[: len(matrix) % folds]] += 1
    demand = capacities[:, None] / len(matrix) * indicators.sum(axis=0)[None, :]
    remaining = np.ones(len(matrix), dtype=bool)
    assignment = np.full(len(matrix), -1, dtype=int)
    while remaining.any():
        counts = indicators[remaining].sum(axis=0)
        positive = np.flatnonzero(counts > 0)
        if len(positive):
            rare = positive[counts[positive] == counts[positive].min()]
            indicator = int(rng.choice(rare))
            subjects = np.flatnonzero(remaining & indicators[:, indicator])
        else:
            indicator = None
            subjects = np.flatnonzero(remaining)
        for subject in rng.permutation(subjects):
            choices = np.flatnonzero(capacities > 0)
            if indicator is not None:
                choices = choices[
                    demand[choices, indicator] == demand[choices, indicator].max()
                ]
                score = demand[choices][:, indicators[subject]].sum(axis=1)
                choices = choices[np.isclose(score, score.max())]
            choices = choices[capacities[choices] == capacities[choices].max()]
            fold = int(rng.choice(choices))
            assignment[subject] = fold
            capacities[fold] -= 1
            demand[fold] -= indicators[subject]
            remaining[subject] = False
    return pd.Series(assignment, index=matrix.index, name="provisional_fold")


def split_diagnostics(
    matrix: pd.DataFrame,
    assignment: pd.Series,
    folds: int,
    min_support: int = EDA_MIN_FOLD_SUPPORT,
) -> pd.DataFrame:
    """Validate an assignment and report every task/fold/class, including zeros."""
    if (
        matrix.index.has_duplicates
        or matrix.index.isna().any()
        or assignment.index.has_duplicates
    ):
        raise ValueError("Subject identifiers must be unique and nonmissing.")
    if len(assignment) != len(matrix) or set(assignment.index) != set(matrix.index):
        raise ValueError("Assignment must contain exactly the subject union.")
    aligned = assignment.reindex(matrix.index)
    if aligned.isna().any() or not aligned.isin(range(folds)).all():
        raise ValueError("Each subject must have one valid fold number.")
    if folds < 2 or set(aligned) != set(range(folds)):
        raise ValueError("Every candidate fold must contain subjects.")
    valid = matrix.apply(valid_labels)
    rows = []
    for fold in range(folds):
        test_mask = aligned.eq(fold)
        train_ids, test_ids = (
            set(matrix.index[~test_mask]),
            set(matrix.index[test_mask]),
        )
        disjoint = train_ids.isdisjoint(test_ids)
        for task in matrix:
            test = valid.loc[test_mask, task]
            train = valid.loc[~test_mask, task]
            total = int(valid[task].notna().sum())
            n_test, n_train = int(test.notna().sum()), int(train.notna().sum())
            for label in EDA_VALID_LABELS:
                test_count, train_count = (
                    int(test.eq(label).sum()),
                    int(train.eq(label).sum()),
                )
                prevalence = (
                    int(valid[task].eq(label).sum()) / total if total else np.nan
                )
                test_proportion = test_count / n_test if n_test else np.nan
                train_proportion = train_count / n_train if n_train else np.nan
                rows.append(
                    dict(
                        task=task,
                        folds=folds,
                        fold=fold,
                        label=label,
                        provisional=True,
                        subject_disjoint=disjoint,
                        train_subjects=len(train_ids),
                        test_subjects=len(test_ids),
                        eligible_train=n_train,
                        eligible_test=n_test,
                        train_count=train_count,
                        test_count=test_count,
                        train_class_missing=train_count == 0,
                        test_class_missing=test_count == 0,
                        tiny_train_support=train_count < min_support,
                        tiny_test_support=test_count < min_support,
                        support_warning_threshold=min_support,
                        full_eligible_proportion=prevalence,
                        test_proportion=test_proportion,
                        train_proportion=train_proportion,
                        test_proportion_difference=test_proportion - prevalence,
                        train_proportion_difference=train_proportion - prevalence,
                    )
                )
    return pd.DataFrame(rows)


def evaluate_candidates(
    tables: list[TaskTable],
    fold_counts: tuple[int, ...] = EDA_FOLD_COUNTS,
    seeds: tuple[int, ...] = EDA_CANDIDATE_SEEDS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return aggregate diagnostics only; keep all candidate assignments local."""
    results, blocked = [], []
    for cohort, group in cohort_tables(tables).items():
        try:
            matrix = subject_label_matrix(group)
        except (ValueError, KeyError) as error:
            blocked.append(dict(cohort=cohort, reason=str(error)))
            continue
        for folds in fold_counts:
            if folds > len(matrix):
                blocked.append(
                    dict(
                        cohort=cohort, reason=f"{folds} folds exceed the subject count."
                    )
                )
                continue
            for seed in seeds:
                assignment = candidate_assignment(matrix, folds, seed)
                diagnostics = split_diagnostics(matrix, assignment, folds)
                diagnostics.insert(0, "cohort", cohort)
                diagnostics.insert(1, "seed", seed)
                results.append(diagnostics)
    return (
        pd.concat(results, ignore_index=True) if results else pd.DataFrame(),
        pd.DataFrame(blocked, columns=["cohort", "reason"]),
    )


def candidate_comparison(diagnostics: pd.DataFrame) -> pd.DataFrame:
    if diagnostics.empty:
        return pd.DataFrame()
    result = diagnostics.groupby(["cohort", "folds", "seed"], sort=False).agg(
        subject_disjoint=("subject_disjoint", "all"),
        missing_test_task_class_folds=("test_class_missing", "sum"),
        missing_train_task_class_folds=("train_class_missing", "sum"),
        worst_test_class_support=("test_count", "min"),
        worst_train_class_support=("train_count", "min"),
        test_subjects_min=("test_subjects", "min"),
        test_subjects_max=("test_subjects", "max"),
    )
    differences = (
        diagnostics.assign(
            absolute_test_difference=diagnostics.test_proportion_difference.abs(),
            absolute_train_difference=diagnostics.train_proportion_difference.abs(),
        )
        .groupby(["cohort", "folds", "seed"], sort=False)[
            ["absolute_test_difference", "absolute_train_difference"]
        ]
        .max()
    )
    return result.join(differences).reset_index()


diagnostics, allocation_warnings = evaluate_candidates(tables)
comparison = candidate_comparison(diagnostics)
display(comparison)
if not allocation_warnings.empty:
    display(allocation_warnings)
if not diagnostics.empty:
    assert diagnostics.subject_disjoint.all()
    assert (comparison.test_subjects_max - comparison.test_subjects_min).le(1).all()
    print(
        "Verified one common assignment per candidate, subject disjointness, and balanced subject counts."
    )

,cohort,folds,seed,subject_disjoint,missing_test_task_class_folds,missing_train_task_class_folds,worst_test_class_support,worst_train_class_support,test_subjects_min,test_subjects_max,absolute_test_difference,absolute_train_difference
0,1year,2,42,True,7,7,0,0,70,70,0.064286,0.064286
1,1year,2,43,True,7,7,0,0,70,70,0.078571,0.078571
2,1year,3,42,True,11,10,0,0,46,47,0.145137,0.073349
3,1year,3,43,True,11,10,0,0,46,47,0.105901,0.051824
4,1year,4,42,True,16,13,0,0,35,35,0.150000,0.050000
5,1year,4,43,True,16,13,0,0,35,35,0.150000,0.050000
6,1year,5,42,True,22,16,0,0,28,28,0.142857,0.035714
7,1year,5,43,True,22,16,0,0,28,28,0.142857,0.035714
8,2year,2,42,True,14,14,0,0,72,73,0.079903,0.079903
9,2year,2,43,True,14,14,0,0,72,73,0.100116,0.100116


Verified one common assignment per candidate, subject disjointness, and balanced subject counts.


In [6]:
if not diagnostics.empty:
    failing = diagnostics.loc[
        diagnostics.test_class_missing | diagnostics.train_class_missing
    ]
    display(
        failing.groupby(["cohort", "folds", "task", "label"], sort=False)
        .agg(
            missing_test_folds_across_seeds=("test_class_missing", "sum"),
            missing_train_folds_across_seeds=("train_class_missing", "sum"),
        )
        .head(20)
    )
    display(
        diagnostics.groupby(["cohort", "folds"], sort=False).agg(
            smallest_eligible_test=("eligible_test", "min"),
            smallest_eligible_train=("eligible_train", "min"),
            worst_test_class_support=("test_count", "min"),
            worst_training_class_support=("train_count", "min"),
        )
    )

missing_test_folds_across_seeds  \
cohort folds task    label                                    
1year  2     Task_1  0                                    4   
                     1                                    4   
                     2                                    4   
             Task_4  0                                    2   
       3     Task_1  0                                    6   
                     1                                    6   
                     2                                    6   
             Task_4  0                                    4   
       4     Task_1  0                                    8   
                     1                                    8   
                     2                                    8   
             Task_4  0                                    6   
             Task_5  0                                    2   
       5     Task_1  0                                   10   
                     1                                   10   
                     2                                   10   
             Task_4  0                                    8   
             Task_5  0                                    4   
             Task_19 0                                    2   
2year  2     Task_1  0                                    4   

                            missing_train_folds_across_seeds  
cohort folds task    label                                    
1year  2     Task_1  0                                     4  
                     1                                     4  
                     2                                     4  
             Task_4  0                                     2  
       3     Task_1  0                                     6  
                     1                                     6  
                     2                                     6  
             Task_4  0                                     2  
       4     Task_1  0                                     8  
                     1                                     8  
                     2                                     8  
             Task_4  0                                     2  
             Task_5  0                                     0  
       5     Task_1  0                                    10  
                     1                                    10  
                     2                                    10  
             Task_4  0                                     2  
             Task_5  0                                     0  
             Task_19 0                                     0  
2year  2     Task_1  0                                     4

smallest_eligible_test  smallest_eligible_train  \
cohort folds                                                    
1year  2                           0                        0   
       3                           0                        0   
       4                           0                        0   
       5                           0                        0   
2year  2                           0                        0   
       3                           0                        0   
       4                           0                        0   
       5                           0                        0   
3year  2                           0                        0   
       3                           0                        0   
       4                           0                        0   
       5                           0                        0   

              worst_test_class_support  worst_training_class_support  
cohort folds                                                          
1year  2                             0                             0  
       3                             0                             0  
       4                             0                             0  
       5                             0                             0  
2year  2                             0                             0  
       3                             0                             0  
       4                             0                             0  
       5                             0                             0  
3year  2                             0                             0  
       3                             0                             0  
       4                             0                             0  
       5                             0                             0

In [7]:
if not feasibility.empty:
    for cohort in feasibility.cohort.unique():
        conclusions = []
        for k in (3, 5):
            subset = task_support.loc[
                task_support.cohort.eq(cohort) & task_support.folds.eq(k)
            ]
            impossible = int((~subset.class_coverage_necessary_condition).sum())
            conclusions.append(
                f"{k}-fold: {impossible} of {len(subset)} tasks fail the necessary class-coverage condition"
            )
        display(Markdown(f"**Observed for {cohort}:** " + "; ".join(conclusions) + "."))

**Observed for 1year:** 3-fold: 2 of 21 tasks fail the necessary class-coverage condition; 5-fold: 4 of 21 tasks fail the necessary class-coverage condition.

**Observed for 2year:** 3-fold: 7 of 21 tasks fail the necessary class-coverage condition; 5-fold: 8 of 21 tasks fail the necessary class-coverage condition.

**Observed for 3year:** 3-fold: 11 of 18 tasks fail the necessary class-coverage condition; 5-fold: 13 of 18 tasks fail the necessary class-coverage condition.